# InterviewSense — Final Communication Model

This notebook is the **next-stage communication model**, replacing the previously audited four-feature baseline.

It is designed to:
- verify the target before training;
- prevent target leakage;
- use speaker-independent validation;
- compare **Ridge Regression** (interpretable primary model) with **Random Forest Regression**;
- report GroupKFold CV and an unseen-speaker test;
- produce actual feature weights/importances;
- run acoustic-only, linguistic-only, and combined ablations.

**Important:** `TARGET_COL` must be a genuine human-rated communication/fluency/interview target. Do not use a mechanically constructed target or one of its component scores as a predictor.

In [ ]:
# Run this only if packages are missing:
# %pip install pandas numpy scikit-learn joblib matplotlib

import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import GroupKFold, GroupShuffleSplit, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
OUTPUT_DIR = Path("InterviewSense_final_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("Environment ready.")

## 1. Dataset configuration
Edit only this cell when using another dataset.

In [ ]:
# ============================================================
# DATASET CONFIGURATION
# ============================================================

DATASET_CSV = r"C:\Users\Mounindra\.cache\kagglehub\datasets\colabsss\multimodal-english-speech-dataset\versions\1\audio_speaking_dataset.csv"

SPEAKER_COL = "speaker_id"

# IMPORTANT: verify this is a genuine human-rated communication /
# fluency / interview target before proceeding.
TARGET_COL = "overall_score"

print("Dataset:", DATASET_CSV)
print("Speaker:", SPEAKER_COL)
print("Target:", TARGET_COL)

## 2. Load and inspect

In [ ]:
df = pd.read_csv(DATASET_CSV)
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
display(df.head())

## 3. Target audit
This prevents repeating the previous `overall_score` leakage problem.

In [ ]:
if TARGET_COL not in df.columns:
    raise RuntimeError(f"Target '{TARGET_COL}' does not exist.")

print("Target:", TARGET_COL)
print("Non-null:", df[TARGET_COL].notna().sum())
print("Unique:", df[TARGET_COL].nunique())

if pd.api.types.is_numeric_dtype(df[TARGET_COL]):
    print("Range:", df[TARGET_COL].min(), "->", df[TARGET_COL].max())
    print("Mean:", df[TARGET_COL].mean())
    print("Median:", df[TARGET_COL].median())

for c in ["pronunciation_score", "fluency_score", "intonation_score"]:
    if c in df.columns:
        print("WARNING: component target present:", c)

print("\nVerify the dataset documentation/annotation protocol before treating this as communication skill.")

## 4. Define safe candidate features

In [ ]:
LEAKAGE_COLUMNS = {
    TARGET_COL, "target", "target_level",
    "pronunciation_score", "fluency_score", "intonation_score",
    "overall_score", "speaker_id", "group_id",
    "file_name", "file_key", "filename", "transcript",
    "audio_path", "filepath"
}

numeric_candidates = [
    c for c in df.columns
    if c not in LEAKAGE_COLUMNS
    and pd.api.types.is_numeric_dtype(df[c])
]

print("Numeric candidates:", len(numeric_candidates))
print(numeric_candidates)

## 5. Acoustic / linguistic feature grouping
Review these lists. Manual override is provided in the next cell.

In [ ]:
ACOUSTIC_KEYWORDS = [
    "duration", "speech_duration", "silence_duration", "speech_ratio",
    "pause", "rms", "energy", "zcr", "pitch", "voiced",
    "centroid", "bandwidth", "mfcc", "jitter", "shimmer",
    "formant", "spectral", "tempo", "intensity"
]

LINGUISTIC_KEYWORDS = [
    "word", "sentence", "syllable", "filler", "grammar",
    "vocab", "lexical", "semantic", "coherence", "cohesion",
    "complexity", "readability", "tfidf", "embedding"
]

def has_kw(col, keywords):
    return any(k in col.lower() for k in keywords)

acoustic_features = [c for c in numeric_candidates if has_kw(c, ACOUSTIC_KEYWORDS)]
linguistic_features = [c for c in numeric_candidates if has_kw(c, LINGUISTIC_KEYWORDS)]

print("ACOUSTIC:", acoustic_features)
print("\nLINGUISTIC:", linguistic_features)

In [ ]:
# ============================================================
# OPTIONAL MANUAL OVERRIDE
# ============================================================
# Replace these lists if the automatic grouping is not correct.
#
# acoustic_features = ["speech_duration_sec", "speaking_rate_wpm", ...]
# linguistic_features = ["word_count", "filler_count", ...]

acoustic_features = [c for c in acoustic_features if c in df.columns]
linguistic_features = [c for c in linguistic_features if c in df.columns]

MODEL_FEATURES = list(dict.fromkeys(acoustic_features + linguistic_features))

if not MODEL_FEATURES:
    raise RuntimeError("No acoustic/linguistic numeric features found. Define them manually.")

print("Final features:", len(MODEL_FEATURES))
print(MODEL_FEATURES)

## 6. Prepare data and validate speakers

In [ ]:
if SPEAKER_COL not in df.columns:
    raise RuntimeError("Speaker ID column is required for speaker-independent validation.")

work = df[[SPEAKER_COL, TARGET_COL] + MODEL_FEATURES].copy()
work[TARGET_COL] = pd.to_numeric(work[TARGET_COL], errors="coerce")
work = work.dropna(subset=[SPEAKER_COL, TARGET_COL]).reset_index(drop=True)
work[SPEAKER_COL] = work[SPEAKER_COL].astype(str)

X = work[MODEL_FEATURES].apply(pd.to_numeric, errors="coerce")
y = work[TARGET_COL]
groups = work[SPEAKER_COL]

usable = [c for c in X.columns if X[c].notna().any()]
X = X[usable]
MODEL_FEATURES = usable
acoustic_features = [c for c in acoustic_features if c in usable]
linguistic_features = [c for c in linguistic_features if c in usable]

print("Usable rows:", len(X))
print("Speakers:", groups.nunique())
print("Features:", len(MODEL_FEATURES))

if groups.nunique() < 6:
    raise RuntimeError("At least 6 speakers are recommended for 5-fold GroupKFold.")

## 7. Speaker-independent train/test split

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx].reset_index(drop=True), X.iloc[test_idx].reset_index(drop=True)
y_train, y_test = y.iloc[train_idx].reset_index(drop=True), y.iloc[test_idx].reset_index(drop=True)
groups_train, groups_test = groups.iloc[train_idx].reset_index(drop=True), groups.iloc[test_idx].reset_index(drop=True)

overlap = set(groups_train) & set(groups_test)

print("Train:", len(X_train), "rows /", groups_train.nunique(), "speakers")
print("Test :", len(X_test), "rows /", groups_test.nunique(), "speakers")
print("Speaker overlap:", len(overlap))
assert not overlap, "SPEAKER LEAKAGE DETECTED!"

## 8. Train and tune Ridge + Random Forest
Ridge is the primary interpretable model; Random Forest is the independent non-linear comparison model.

In [ ]:
gkf = GroupKFold(n_splits=5)

ridge_search = GridSearchCV(
    Pipeline([("scaler", StandardScaler()), ("model", Ridge())]),
    {"model__alpha": [0.01, 0.1, 1, 10, 100, 1000]},
    scoring="neg_root_mean_squared_error",
    cv=gkf, n_jobs=-1
)
ridge_search.fit(X_train, y_train, groups=groups_train)

rf_search = GridSearchCV(
    RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1),
    {
        "n_estimators": [200],
        "max_depth": [None, 8, 15],
        "min_samples_leaf": [1, 3, 5],
        "max_features": ["sqrt", 1.0]
    },
    scoring="neg_root_mean_squared_error",
    cv=gkf, n_jobs=-1
)
rf_search.fit(X_train, y_train, groups=groups_train)

ridge_model = ridge_search.best_estimator_
rf_model = rf_search.best_estimator_

print("Best Ridge:", ridge_search.best_params_)
print("Best RF:", rf_search.best_params_)

## 9. GroupKFold metrics

In [ ]:
def cv_metrics(model_factory, X, y, groups):
    maes, rmses, r2s = [], [], []
    for fold, (tr, va) in enumerate(gkf.split(X, y, groups), 1):
        model = model_factory()
        model.fit(X.iloc[tr], y.iloc[tr])
        pred = model.predict(X.iloc[va])
        maes.append(mean_absolute_error(y.iloc[va], pred))
        rmses.append(np.sqrt(mean_squared_error(y.iloc[va], pred)))
        r2s.append(r2_score(y.iloc[va], pred))
        print(f"Fold {fold}: MAE={maes[-1]:.4f}, RMSE={rmses[-1]:.4f}, R2={r2s[-1]:.4f}")
    return {
        "MAE_mean": np.mean(maes), "MAE_std": np.std(maes),
        "RMSE_mean": np.mean(rmses), "RMSE_std": np.std(rmses),
        "R2_mean": np.mean(r2s), "R2_std": np.std(r2s)
    }

ridge_cv = cv_metrics(
    lambda: Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=ridge_search.best_params_["model__alpha"]))
    ]),
    X_train, y_train, groups_train
)

rf_params = rf_search.best_params_.copy()
ridge_cv_row = {"Model": "Ridge", **ridge_cv}

rf_cv = cv_metrics(
    lambda: RandomForestRegressor(
        **rf_params, random_state=RANDOM_SEED, n_jobs=-1
    ),
    X_train, y_train, groups_train
)
rf_cv_row = {"Model": "Random Forest", **rf_cv}

cv_table = pd.DataFrame([ridge_cv_row, rf_cv_row])
display(cv_table)
cv_table.to_csv(OUTPUT_DIR / "groupkfold_model_comparison.csv", index=False)

## 10. Held-out unseen-speaker test

In [ ]:
def evaluate(model, Xtr, ytr, Xte, yte):
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    return {
        "MAE": mean_absolute_error(yte, pred),
        "RMSE": np.sqrt(mean_squared_error(yte, pred)),
        "R2": r2_score(yte, pred),
        "pred": pred
    }

ridge_test = evaluate(ridge_model, X_train, y_train, X_test, y_test)
rf_test = evaluate(rf_model, X_train, y_train, X_test, y_test)

test_table = pd.DataFrame([
    {"Model": "Ridge", "MAE": ridge_test["MAE"], "RMSE": ridge_test["RMSE"], "R2": ridge_test["R2"]},
    {"Model": "Random Forest", "MAE": rf_test["MAE"], "RMSE": rf_test["RMSE"], "R2": rf_test["R2"]}
])
display(test_table)
test_table.to_csv(OUTPUT_DIR / "heldout_test_comparison.csv", index=False)

## 11. Actual learned feature weights

In [ ]:
ridge_weights = pd.DataFrame({
    "Feature": MODEL_FEATURES,
    "Standardized_Ridge_weight": ridge_model.named_steps["model"].coef_
})
ridge_weights["Absolute_weight"] = ridge_weights["Standardized_Ridge_weight"].abs()
ridge_weights = ridge_weights.sort_values("Absolute_weight", ascending=False)
display(ridge_weights)
ridge_weights.to_csv(OUTPUT_DIR / "ridge_feature_weights.csv", index=False)

rf_importance = pd.DataFrame({
    "Feature": MODEL_FEATURES,
    "RF_importance": rf_model.feature_importances_
}).sort_values("RF_importance", ascending=False)
display(rf_importance)
rf_importance.to_csv(OUTPUT_DIR / "random_forest_feature_importance.csv", index=False)

## 12. Acoustic vs linguistic ablation

In [ ]:
def ablation_cv(features):
    features = [f for f in features if f in X_train.columns]
    if not features:
        return [np.nan, np.nan, np.nan]
    maes, rmses, r2s = [], [], []
    for tr, va in gkf.split(X_train[features], y_train, groups_train):
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=ridge_search.best_params_["model__alpha"]))
        ])
        model.fit(X_train[features].iloc[tr], y_train.iloc[tr])
        pred = model.predict(X_train[features].iloc[va])
        maes.append(mean_absolute_error(y_train.iloc[va], pred))
        rmses.append(np.sqrt(mean_squared_error(y_train.iloc[va], pred)))
        r2s.append(r2_score(y_train.iloc[va], pred))
    return [np.mean(maes), np.mean(rmses), np.mean(r2s)]

ablation = pd.DataFrame([
    ["Acoustic only", len(acoustic_features), *ablation_cv(acoustic_features)],
    ["Linguistic only", len(linguistic_features), *ablation_cv(linguistic_features)],
    ["Acoustic + Linguistic", len(MODEL_FEATURES), *ablation_cv(MODEL_FEATURES)]
], columns=["Feature_set", "Feature_count", "MAE", "RMSE", "R2"])

display(ablation)
ablation.to_csv(OUTPUT_DIR / "ablation_results.csv", index=False)

## 13. Visual evaluation

In [ ]:
pred = ridge_test["pred"]
plt.figure(figsize=(7,6))
plt.scatter(y_test, pred, alpha=0.6)
lo = min(y_test.min(), pred.min())
hi = max(y_test.max(), pred.max())
plt.plot([lo, hi], [lo, hi], linestyle="--")
plt.xlabel("Actual score")
plt.ylabel("Predicted score")
plt.title("Ridge — Unseen Speaker Test")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "ridge_predicted_vs_actual.png", dpi=200)
plt.show()

residuals = y_test.values - pred
plt.figure(figsize=(8,5))
plt.hist(residuals, bins=30)
plt.xlabel("Residual")
plt.ylabel("Count")
plt.title("Ridge Residual Distribution")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "ridge_residuals.png", dpi=200)
plt.show()

## 14. Final leakage/integrity checks

In [ ]:
for feature in MODEL_FEATURES:
    assert feature not in LEAKAGE_COLUMNS, f"Leakage feature: {feature}"

assert not (set(groups_train) & set(groups_test)), "Speaker leakage!"
assert not {"pronunciation_score", "fluency_score", "intonation_score", "overall_score"}.intersection(MODEL_FEATURES)

constant_features = [c for c in MODEL_FEATURES if X_train[c].nunique(dropna=True) <= 1]
print("Constant features:", constant_features)
print("Speaker overlap:", len(set(groups_train) & set(groups_test)))
print("ALL STRUCTURAL LEAKAGE CHECKS PASSED.")

## 15. Save final artifacts

In [ ]:
joblib.dump(ridge_model, OUTPUT_DIR / "InterviewSense_Ridge_final.joblib")

metadata = {
    "target": TARGET_COL,
    "speaker_column": SPEAKER_COL,
    "features": MODEL_FEATURES,
    "acoustic_features": acoustic_features,
    "linguistic_features": linguistic_features,
    "primary_model": "Ridge",
    "comparison_model": "RandomForestRegressor",
    "validation": "GroupKFold + unseen-speaker holdout",
    "random_seed": RANDOM_SEED
}
with open(OUTPUT_DIR / "model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Artifacts saved to:", OUTPUT_DIR.resolve())
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name)

## Final interpretation

A successful run gives you:

**Audio/acoustic + linguistic features → Ridge communication model**

and an independent:

**same features → Random Forest comparison**

with **speaker-independent GroupKFold** and a held-out unseen-speaker test.

The Ridge coefficients are the requested learned weights. Random Forest importances provide an independent feature-importance comparison.

Do **not** claim that a high score proves real-world communication ability unless the dataset's human annotation protocol supports that claim.